# Testing Whether Competition Dataset is a Disguised Transform of Training Dataset

**Objective**: Build a rigorous test to determine whether the competition dataset's raw satellite bands were produced by applying a deterministic, disguising transform to the training dataset (or data from the same underlying population) — specifically a circular shift of the month index plus a per-band affine rescaling (value' = a_b * value + b_b) — as opposed to being genuinely independent measurements from a different geographic region.

We will test both hypotheses:
- **H0 (independent regions)**: Training and competition rows come from unrelated geographic areas. Any similarity is coincidental or reflects broadly shared climate/seasonal physics.
- **H1 (disguised same-origin data)**: Competition rows are training rows (or from the same underlying population) relabeled with a shifted month index and rescaled per band.

We will actively look for disconfirming evidence, not just confirmation, and give a calibrated verdict.


In [ ]:
# Step 0: Metadata Scan
print("=== STEP 0: METADATA SCAN ===")
print()

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import pearsonr, spearmanr
import seaborn as sns
from sklearn.linear_model import LinearRegression


# Check file sizes
train_path = '../data/Train.csv'
test_path = '../data/Test.csv'

print(f"Training data file: {train_path}")
print(f"Competition data file: {test_path}")
print()

if os.path.exists(train_path) and os.path.exists(test_path):
    train_size = os.path.getsize(train_path) / (1024 * 1024)  # MB
    test_size = os.path.getsize(test_path) / (1024 * 1024)  # MB
    print(f"File sizes:")
    print(f"  Training: {train_size:.2f} MB")
    print(f"  Competition: {test_size:.2f} MB")
    print()
else:
    print("ERROR: One or both data files not found!")
    print()

# Load small samples to check structure
print("Loading small samples to examine structure...")
train_sample = pd.read_csv(train_path, nrows=5)
test_sample = pd.read_csv(test_path, nrows=5)

print("Training data columns (first 20):")
print(list(train_sample.columns)[:20])
print("...")
print(f"Total columns: {len(train_sample.columns)}")
print()

print("Competition data columns (first 20):")
print(list(test_sample.columns)[:20])
print("...")
print(f"Total columns: {len(test_sample.columns)}")
print()

# Check if they have the same columns
train_cols = set(train_sample.columns)
test_cols = set(test_sample.columns)
if train_cols == test_cols:
    print("✓ Both datasets have identical column structure")
else:
    print("✗ Column structures differ!")
    print(f"  Only in train: {train_cols - test_cols}")
    print(f"  Only in test: {test_cols - train_cols}")
print()

In [ ]:
# Step 1: Load & characterize
print("=== STEP 1: LOAD & CHARACTERIZE ===")
print()

# Load full datasets
print("Loading full datasets...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Dataset shapes:")
print(f"  Training: {train_df.shape}")
print(f"  Competition: {test_df.shape}")
print()

# Confirm the 12-band x 12-month structure
# Expected columns: ID, label (for train only), then bands for each month
# Bands: VH, VV, blue, green, nir, nira, re1, re2, re3, red, swir1, swir2 (12 bands)
# Months: 01-12 (12 months)
# So we expect: 1 (ID) + 1 (label, train only) + 12*12 = 146 columns for train
#             1 (ID) + 12*12 = 145 columns for test

print("Column count verification:")
print(f"  Training columns: {len(train_df.columns)} (expected: 146)")
print(f"  Competition columns: {len(test_df.columns)} (expected: 145)")
print()

# Extract band names and month numbers from column names (excluding ID and label)
def extract_bands_and_months(df_columns):
    """Extract unique band names and month numbers from column names"""
    bands = set()
    months = set()
    
    for col in df_columns:
        if col not in ['ID', 'label']:  # Skip ID and label columns
            # Format is typically {band}_{month} or {band}_{month:02d}
            parts = col.split('_')
            if len(parts) >= 2:
                band = parts[0]
                month_part = parts[1]
                # Handle both formats: "01" and "1" 
                if month_part.isdigit():
                    months.add(int(month_part))
                bands.add(band)
    
    return sorted(list(bands)), sorted(list(months))

train_bands, train_months = extract_bands_and_months(train_df.columns)
test_bands, test_months = extract_bands_and_months(test_df.columns)

print(f"Bands found in training: {train_bands}")
print(f"Months found in training: {train_months}")
print()
print(f"Bands found in competition: {test_bands}")
print(f"Months found in competition: {test_months}")
print()

# Check consistency
if set(train_bands) == set(test_bands) and set(train_months) == set(test_months):
    print("✓ Band and month structure is consistent between datasets")
else:
    print("✗ Band/month structure differs between datasets!")
    print(f"  Band difference: {set(train_bands) ^ set(test_bands)}")
    print(f"  Month difference: {set(train_months) ^ set(test_months)}")
print()

# Define the standard bands and months we expect
expected_bands = ['VH', 'VV', 'blue', 'green', 'nir', 'nira', 're1', 're2', 're3', 'red', 'swir1', 'swir2']
expected_months = [f"{i:02d}" for i in range(1, 13)]  # ['01', '02', ..., '12']

print(f"Expected bands: {expected_bands}")
print(f"Expected months: {expected_months}")
print()

# Check if we have the expected structure
missing_bands_train = set(expected_bands) - set(train_bands)
missing_months_train = set(expected_months) - set([f"{m:02d}" for m in train_months])
missing_bands_test = set(expected_bands) - set(test_bands)
missing_months_test = set(expected_months) - set([f"{m:02d}" for m in test_months])

if not missing_bands_train and not missing_months_train and not missing_bands_test and not missing_months_test:
    print("✓ All expected bands and months are present")
else:
    if missing_bands_train:
        print(f"✗ Missing bands in training: {missing_bands_train}")
    if missing_months_train:
        print(f"✗ Missing months in training: {missing_months_train}")
    if missing_bands_test:
        print(f"✗ Missing bands in competition: {missing_bands_test}")
    if missing_months_test:
        print(f"✗ Missing months in competition: {missing_months_test}")
print()

# Report: row counts for each dataset
print(f"Row counts:")
print(f"  Training rows: {len(train_df):,}")
print(f"  Competition rows: {len(test_df):,}")
print()

# Distribution of "number of months observed per row"
def count_observed_months_per_row(df_row, bands, months_format):
    """Count how many months have at least one band with valid data"""
    observed_count = 0
    for month in months_format:
        month_has_data = False
        for band in bands:
            feature = f"{band}_{month}"
            if feature in df_row and not pd.isna(df_row[feature]) and df_row[feature] != -9999:
                month_has_data = True
                break
        if month_has_data:
            observed_count += 1
    return observed_count

print("Computing months observed per row...")
train_observed_counts = []
test_observed_counts = []

# Use expected format for month strings
months_str_format = [f"{i:02d}" for i in range(1, 13)]

for idx, row in train_df.iterrows():
    count = count_observed_months_per_row(row, expected_bands, months_str_format)
    train_observed_counts.append(count)
    
for idx, row in test_df.iterrows():
    count = count_observed_months_per_row(row, expected_bands, months_str_format)
    test_observed_counts.append(count)

train_observed_counts = np.array(train_observed_counts)
test_observed_counts = np.array(test_observed_counts)

print(f"Months observed per row - Training:")
print(f"  Mean: {np.mean(train_observed_counts):.2f}")
print(f"  Median: {np.median(train_observed_counts):.0f}")
print(f"  Min: {np.min(train_observed_counts)}")
print(f"  Max: {np.max(train_observed_counts)}")
print(f"  Distribution: {np.bincount(train_observed_counts)[1:]}")  # Skip 0 count
print()

print(f"Months observed per row - Competition:")
print(f"  Mean: {np.mean(test_observed_counts):.2f}")
print(f"  Median: {np.median(test_observed_counts):.0f}")
print(f"  Min: {np.min(test_observed_counts)}")
print(f"  Max: {np.max(test_observed_counts)}")
print(f"  Distribution: {np.bincount(test_observed_counts)[1:]}")  # Skip 0 count
print()

# Establish missing-value convention
print("Establishing missing-value convention...")
print("Current missing value indicators:")
print(f"  -9999: Present in both datasets")
print(f"  NaN: {train_df.isna().any().any()} in training, {test_df.isna().any().any()} in competition")

# Convert -9999 to NaN for consistent handling
print("\nConverting -9999 values to NaN for consistent handling...")
train_df_clean = train_df.replace(-9999, np.nan)
test_df_clean = test_df.replace(-9999, np.nan)

print(f"After conversion:")
print(f"  Training NaN count: {train_df_clean.isna().sum().sum():,}")
print(f"  Competition NaN count: {test_df_clean.isna().sum().sum():,}")
print()

# Per-band per-month summary stats (min, median, max) for both datasets side by side
print("Computing per-band per-month summary statistics...")
print("(This may take a moment for the full datasets)")

# Prepare summary statistics
summary_data = []

for band in expected_bands:
    for month in months_str_format:
        feature = f"{band}_{month}"
        
        if feature in train_df_clean.columns and feature in test_df_clean.columns:
            # Training stats
            train_vals = train_df_clean[feature].dropna()
            train_min = train_vals.min() if len(train_vals) > 0 else np.nan
            train_median = train_vals.median() if len(train_vals) > 0 else np.nan
            train_max = train_vals.max() if len(train_vals) > 0 else np.nan
            
            # Competition stats
            test_vals = test_df_clean[feature].dropna()
            test_min = test_vals.min() if len(test_vals) > 0 else np.nan
            test_median = test_vals.median() if len(test_vals) > 0 else np.nan
            test_max = test_vals.max() if len(test_vals) > 0 else np.nan
            
            summary_data.append({
                'band': band,
                'month': month,
                'train_min': train_min,
                'train_median': train_median,
                'train_max': train_max,
                'test_min': test_min,
                'test_median': test_median,
                'test_max': test_max
            })

summary_df = pd.DataFrame(summary_data)

print("Summary statistics computed.")
print()

# Show a few examples
print("Example summary statistics (first 6 band-month combinations):")
display_cols = ['band', 'month', 'train_min', 'train_median', 'train_max', 
                'test_min', 'test_median', 'test_max']
print(summary_df[display_cols].head(6).to_string(index=False))
print()

# Save the cleaned dataframes for use in subsequent steps
print("Step 1 complete. Cleaned dataframes saved for subsequent steps.")
train_df_features = train_df_clean.copy()
test_df_features = test_df_clean.copy()
# We'll keep the original IDs and labels separate
if 'label' in train_df.columns:
    train_labels = train_df_clean['label'].copy()
    train_df_features = train_df_features.drop('label', axis=1)
else:
    train_labels = None
test_ids = test_df_clean['ID'].copy()
train_ids = train_df_clean['ID'].copy()
test_df_features = test_df_features.drop('ID', axis=1)
train_df_features = train_df_features.drop('ID', axis=1)

print(f"Features-only shapes:")
print(f"  Training features: {train_df_features.shape}")
print(f"  Competition features: {test_df_features.shape}")
print()

In [ ]:
# Step 2: Aggregate cyclic-shift scan (cheap first test)
print("=== STEP 2: AGGREGATE CYCLIC-SHIFT SCAN ===")
print()

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Computing monthly median values for each band...")
print("(Using NaN-aware median computation)")

# Define bands and months
bands = expected_bands  # ['VH', 'VV', 'blue', 'green', 'nir', 'nira', 're1', 're2', 're3', 'red', 'swir1', 'swir2']
months = [f"{i:02d}" for i in range(1, 13)]  # ['01', '02', ..., '12']

# Compute median value across all rows for each band-month combination
def compute_monthly_medians(df, bands, months):
    """Compute median for each band-month, ignoring NaN values"""
    medians = np.full((len(bands), len(months)), np.nan)
    
    for b_idx, band in enumerate(bands):
        for m_idx, month in enumerate(months):
            feature = f"{band}_{month}"
            if feature in df.columns:
                vals = df[feature].dropna()
                if len(vals) > 0:
                    medians[b_idx, m_idx] = np.median(vals)
    
    return medians

print("Computing training medians...")
train_medians = compute_monthly_medians(train_df_features, bands, months)
print("Computing competition medians...")
test_medians = compute_monthly_medians(test_df_features, bands, months)

print("Medians computed.")
print()

# For each candidate integer shift τ = 0..11:
#   • Circularly shift the competition curve by τ months.
#   • Compute Pearson and Spearman correlation against the training curve, per band.
#   • Average across all 12 bands to get one aggregate score per τ.

print("Testing cyclic shifts τ = 0..11...")
pearson_scores = []
spearman_scores = []
per_band_pearson = []  # Store per-band correlations for best τ
per_band_spearman = []

for tau in range(12):
    band_pearson_corrs = []
    band_spearman_corrs = []
    
    for b_idx, band in enumerate(bands):
        # Get the 12-point curve for this band
        train_curve = train_medians[b_idx, :]  # Shape: (12,)
        test_curve = test_medians[b_idx, :]    # Shape: (12,)
        
        # Circularly shift competition curve by τ months
        # Positive tau means shifting test curve to the left (earlier months)
        shifted_test_curve = np.roll(test_curve, -tau)
        
        # Compute correlations, ignoring NaN pairs
        valid_mask = ~(np.isnan(train_curve) | np.isnan(shifted_test_curve))
        if np.sum(valid_mask) >= 2:  # Need at least 2 valid points
            train_valid = train_curve[valid_mask]
            test_valid = shifted_test_curve[valid_mask]
            
            # Pearson correlation
            if len(train_valid) > 1 and np.std(train_valid) > 0 and np.std(test_valid) > 0:
                pearson_corr, _ = pearsonr(train_valid, test_valid)
                band_pearson_corrs.append(pearson_corr)
            else:
                band_pearson_corrs.append(np.nan)
                
            # Spearman correlation
            if len(train_valid) > 1:
                spearman_corr, _ = spearmanr(train_valid, test_valid)
                band_spearman_corrs.append(spearman_corr)
            else:
                band_spearman_corrs.append(np.nan)
        else:
            band_pearson_corrs.append(np.nan)
            band_spearman_corrs.append(np.nan)
    
    # Average across bands (ignoring NaN)
    avg_pearson = np.nanmean(band_pearson_corrs) if len(band_pearson_corrs) > 0 else np.nan
    avg_spearman = np.nanmean(band_spearman_corrs) if len(band_spearman_corrs) > 0 else np.nan
    
    pearson_scores.append(avg_pearson)
    spearman_scores.append(avg_spearman)
    
    # Store for best tau later
    if tau == 0:  # Initialize for tau=0
        per_band_pearson = band_pearson_corrs.copy()
        per_band_spearman = band_spearman_corrs.copy()
    elif not np.isnan(avg_pearson) and (np.isnan(np.nanmean(per_band_pearson)) or avg_pearson > np.nanmean(per_band_pearson)):
        per_band_pearson = band_pearson_corrs.copy()
        per_band_spearman = band_spearman_corrs.copy()

# Find the best tau* (maximizing aggregate Pearson correlation)
valid_pearson = [(i, score) for i, score in enumerate(pearson_scores) if not np.isnan(score)]
if valid_pearson:
    best_tau, best_avg_pearson = max(valid_pearson, key=lambda x: x[1])
else:
    best_tau, best_avg_pearson = 0, np.nan

print(f"Results:")
print(f"  Best shift τ*: {best_tau}")
print(f"  Best average Pearson correlation: {best_avg_pearson:.4f}")
print(f"  Best average Spearman correlation: {spearman_scores[best_tau]:.4f}")
print()

# Show all tau scores
print("Aggregate correlation vs. shift τ:")
print("τ\tPearson\t\tSpearman")
print("-" * 30)
for tau in range(12):
    p_score = pearson_scores[tau] if not np.isnan(pearson_scores[tau]) else float('nan')
    s_score = spearman_scores[tau] if not np.isnan(spearman_scores[tau]) else float('nan')
    print(f"{tau}\t{p_score:.4f}\t\t{s_score:.4f}")
print()

# Create plot: aggregate correlation vs. shift τ (each band as a thin line, the mean as a bold line)
plt.figure(figsize=(12, 8))

# Plot individual band correlations as thin lines
colors = plt.cm.tab10(np.linspace(0, 1, len(bands)))
for b_idx, band in enumerate(bands):
    band_pearson_taus = []
    for tau in range(12):
        # Recompute or retrieve band-specific correlation for this tau
        train_curve = train_medians[b_idx, :]
        test_curve = test_medians[b_idx, :]
        shifted_test_curve = np.roll(test_curve, -tau)
        
        valid_mask = ~(np.isnan(train_curve) | np.isnan(shifted_test_curve))
        if np.sum(valid_mask) >= 2:
            train_valid = train_curve[valid_mask]
            test_valid = shifted_test_curve[valid_mask]
            if len(train_valid) > 1 and np.std(train_valid) > 0 and np.std(test_valid) > 0:
                corr, _ = pearsonr(train_valid, test_valid)
                band_pearson_taus.append(corr)
            else:
                band_pearson_taus.append(np.nan)
        else:
            band_pearson_taus.append(np.nan)
    
    plt.plot(range(12), band_pearson_taus, color=colors[b_idx], alpha=0.3, linewidth=0.5, label=band if b_idx < 3 else "")

# Plot the mean as a bold line
plt.plot(range(12), pearson_scores, color='black', linewidth=3, label='Mean (Pearson)')
plt.plot(range(12), spearman_scores, color='red', linewidth=3, linestyle='--', label='Mean (Spearman)')

# Mark the best tau
plt.axvline(x=best_tau, color='green', linestyle=':', linewidth=2, label=f'Best τ* = {best_tau}')

plt.xlabel('Shift τ (months)', fontsize=12)
plt.ylabel('Correlation Coefficient', fontsize=12)
plt.title('Aggregate Correlation vs. Cyclic Shift τ\n(Thin lines: individual bands; Bold lines: mean across bands)', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xticks(range(12))
plt.tight_layout()
plt.show()

# Table of per-band correlation at whichever τ maximizes the aggregate score
print(f"Per-band correlations at best τ* = {best_tau}:")
print("Band\t\tPearson\t\tSpearman")
print("-" * 40)
for b_idx, band in enumerate(bands):
    p_corr = per_band_pearson[b_idx] if b_idx < len(per_band_pearson) and not np.isnan(per_band_pearson[b_idx]) else np.nan
    s_corr = per_band_spearman[b_idx] if b_idx < len(per_band_spearman) and not np.isnan(per_band_spearman[b_idx]) else np.nan
    print(f"{band:<12}\t{p_corr:.4f}\t\t{s_corr:.4f}")
print()

# Save results for subsequent steps
print("Step 2 complete. Results saved for subsequent steps.")
step2_results = {
    'best_tau': best_tau,
    'best_avg_pearson': best_avg_pearson,
    'pearson_scores': pearson_scores,
    'spearman_scores': spearman_scores,
    'per_band_pearson_at_best_tau': per_band_pearson,
    'per_band_spearman_at_best_tau': per_band_spearman,
    'train_medians': train_medians,
    'test_medians': test_medians
}

In [ ]:
# Step 3: Calibrate against two reference baselines
print("=== STEP 3: CALIBRATE AGAINST TWO REFERENCE BASELINES ===")
print()

print("Running Step 2 procedure on reference baselines...")
print()

# Same-distribution ceiling: randomly split the training set in half
print("Creating same-distribution baseline (random split of training set)...")
n_train = len(train_df_features)
split_idx = n_train // 2

# Shuffle and split
train_indices = np.random.permutation(n_train)
train_indices_shuffled = train_indices  # Already shuffled from permutation
train_half1_idx = train_indices_shuffled[:split_idx]
train_half2_idx = train_indices_shuffled[split_idx:split_idx*2]  # Ensure equal size

train_half1 = train_df_features.iloc[train_half1_idx].reset_index(drop=True)
train_half2 = train_df_features.iloc[train_half2_idx].reset_index(drop=True)

print(f"Half 1 size: {len(train_half1)}")
print(f"Half 2 size: {len(train_half2)}")
print()

# Function to compute medians for a dataframe (reusing from Step 2)
def compute_monthly_medians_for_df(df, bands, months):
    medians = np.full((len(bands), len(months)), np.nan)
    for b_idx, band in enumerate(bands):
        for m_idx, month in enumerate(months):
            feature = f"{band}_{month}"
            if feature in df.columns:
                vals = df[feature].dropna()
                if len(vals) > 0:
                    medians[b_idx, m_idx] = np.median(vals)
    return medians

# Compute medians for each half
print("Computing medians for training halves...")
medians_half1 = compute_monthly_medians_for_df(train_half1, bands, months)
medians_half2 = compute_monthly_medians_for_df(train_half2, bands, months)

# Run Step 2 procedure between the two halves
print("Running cyclic-shift scan between training halves...")
same_dist_pearson = []
same_dist_spearman = []

for tau in range(12):
    band_pearson_corrs = []
    band_spearman_corrs = []
    
    for b_idx, band in enumerate(bands):
        curve1 = medians_half1[b_idx, :]
        curve2 = medians_half2[b_idx, :]
        shifted_curve2 = np.roll(curve2, -tau)
        
        valid_mask = ~(np.isnan(curve1) | np.isnan(shifted_curve2))
        if np.sum(valid_mask) >= 2:
            curve1_valid = curve1[valid_mask]
            curve2_valid = shifted_curve2[valid_mask]
            if len(curve1_valid) > 1 and np.std(curve1_valid) > 0 and np.std(curve2_valid) > 0:
                pearson_corr, _ = pearsonr(curve1_valid, curve2_valid)
                spearman_corr, _ = spearmanr(curve1_valid, curve2_valid)
                band_pearson_corrs.append(pearson_corr)
                band_spearman_corrs.append(spearman_corr)
            else:
                band_pearson_corrs.append(np.nan)
                band_spearman_corrs.append(np.nan)
        else:
            band_pearson_corrs.append(np.nan)
            band_spearman_corrs.append(np.nan)
    
    avg_pearson = np.nanmean(band_pearson_corrs) if len(band_pearson_corrs) > 0 else np.nan
    avg_spearman = np.nanmean(band_spearman_corrs) if len(band_spearman_corrs) > 0 else np.nan
    
    same_dist_pearson.append(avg_pearson)
    same_dist_spearman.append(avg_spearman)

# Find best tau for same-distribution baseline
valid_same_dist = [(i, score) for i, score in enumerate(same_dist_pearson) if not np.isnan(score)]
if valid_same_dist:
    best_tau_same_dist, best_avg_pearson_same_dist = max(valid_same_dist, key=lambda x: x[1])
else:
    best_tau_same_dist, best_avg_pearson_same_dist = 0, np.nan

print(f"Same-distribution baseline results:")
print(f"  Best shift τ: {best_tau_same_dist}")
print(f"  Best average Pearson correlation: {best_avg_pearson_same_dist:.4f}")
print(f"  Best average Spearman correlation: {same_dist_spearman[best_tau_same_dist]:.4f}")
print()

# Unrelated floor: run Step 2 between training and a version of competition with row identities randomly shuffled
print("Creating unrelated baseline (competition with shuffled row identities)...")
n_test = len(test_df_features)
test_indices_shuffled = np.random.permutation(n_test)
test_shuffled = test_df_features.iloc[test_indices_shuffled].reset_index(drop=True)

print(f"Original competition size: {len(test_df_features)}")
print(f"Shuffled competition size: {len(test_shuffled)}")
print()

# Compute medians for shuffled competition
print("Computing medians for shuffled competition...")
medians_test_shuffled = compute_monthly_medians_for_df(test_shuffled, bands, months)

# Run Step 2 procedure between training and shuffled competition
print("Running cyclic-shift scan between training and shuffled competition...")
unrelated_pearson = []
unrelated_spearman = []

for tau in range(12):
    band_pearson_corrs = []
    band_spearman_corrs = []
    
    for b_idx, band in enumerate(bands):
        train_curve = train_medians[b_idx, :]  # From Step 2
        test_curve = medians_test_shuffled[b_idx, :]
        shifted_test_curve = np.roll(test_curve, -tau)
        
        valid_mask = ~(np.isnan(train_curve) | np.isnan(shifted_test_curve))
        if np.sum(valid_mask) >= 2:
            train_valid = train_curve[valid_mask]
            test_valid = shifted_test_curve[valid_mask]
            if len(train_valid) > 1 and np.std(train_valid) > 0 and np.std(test_valid) > 0:
                pearson_corr, _ = pearsonr(train_valid, test_valid)
                spearman_corr, _ = spearmanr(train_valid, test_valid)
                band_pearson_corrs.append(pearson_corr)
                band_spearman_corrs.append(spearman_corr)
            else:
                band_pearson_corrs.append(np.nan)
                band_spearman_corrs.append(np.nan)
        else:
            band_pearson_corrs.append(np.nan)
            band_spearman_corrs.append(np.nan)
    
    avg_pearson = np.nanmean(band_pearson_corrs) if len(band_pearson_corrs) > 0 else np.nan
    avg_spearman = np.nanmean(band_spearman_corrs) if len(band_spearman_corrs) > 0 else np.nan
    
    unrelated_pearson.append(avg_pearson)
    unrelated_spearman.append(avg_spearman)

# Repeat the unrelated baseline multiple times to get a distribution
print("Running multiple iterations for unrelated baseline distribution...")
n_iterations = 20
unrelated_pearson_all = []
unrelated_spearman_all = []

for iteration in range(n_iterations):
    if iteration % 5 == 0:
        print(f"  Iteration {iteration+1}/{n_iterations}")
    
    # Shuffle competition rows
    test_indices_shuffled = np.random.permutation(n_test)
    test_shuffled_iter = test_df_features.iloc[test_indices_shuffled].reset_index(drop=True)
    
    # Compute medians
    medians_test_shuffled_iter = compute_monthly_medians_for_df(test_shuffled_iter, bands, months)
    
    # Compute correlations for this iteration
    iter_pearson = []
    iter_spearman = []
    
    for tau in range(12):
        band_pearson_corrs = []
        band_spearman_corrs = []
        
        for b_idx, band in enumerate(bands):
            train_curve = train_medians[b_idx, :]
            test_curve = medians_test_shuffled_iter[b_idx, :]
            shifted_test_curve = np.roll(test_curve, -tau)
            
            valid_mask = ~(np.isnan(train_curve) | np.isnan(shifted_test_curve))
            if np.sum(valid_mask) >= 2:
                train_valid = train_curve[valid_mask]
                test_valid = shifted_test_curve[valid_mask]
                if len(train_valid) > 1 and np.std(train_valid) > 0 and np.std(test_valid) > 0:
                    pearson_corr, _ = pearsonr(train_valid, test_valid)
                    spearman_corr, _ = spearmanr(train_valid, test_valid)
                    band_pearson_corrs.append(pearson_corr)
                    band_spearman_corrs.append(spearman_corr)
                else:
                    band_pearson_corrs.append(np.nan)
                    band_spearman_corrs.append(np.nan)
            else:
                band_pearson_corrs.append(np.nan)
                band_spearman_corrs.append(np.nan)
        
        avg_pearson = np.nanmean(band_pearson_corrs) if len(band_pearson_corrs) > 0 else np.nan
        avg_spearman = np.nanmean(band_spearman_corrs) if len(band_spearman_corrs) > 0 else np.nan
        
        iter_pearson.append(avg_pearson)
        iter_spearman.append(avg_spearman)
    
    unrelated_pearson_all.append(iter_pearson)
    unrelated_spearman_all.append(iter_spearman)

unrelated_pearson_all = np.array(unrelated_pearson_all)  # Shape: (n_iterations, 12)
unrelated_spearman_all = np.array(unrelated_spearman_all)  # Shape: (n_iterations, 12)

print("Unrelated baseline distribution computed.")
print()

# Report where the real train-vs-competition result from Step 2 falls relative to these two anchors
print("=== BASELINE COMPARISON ===")
print(f"Real train-vs-competition (from Step 2):")
print(f"  Best τ*: {step2_results['best_tau']}")
print(f"  Best Pearson: {step2_results['best_avg_pearson']:.4f}")
print(f"  Best Spearman: {step2_results['spearman_scores'][step2_results['best_tau']]:.4f}")
print()

print(f"Same-distribution ceiling (training vs. training half-split):")
print(f"  Best τ: {best_tau_same_dist}")
print(f"  Best Pearson: {best_avg_pearson_same_dist:.4f}")
print(f"  Best Spearman: {same_dist_spearman[best_tau_same_dist]:.4f}")
print()

print(f"Unrelated floor distribution (training vs. shuffled competition):")
print(f"  Pearson - Mean: {np.nanmean(unrelated_pearson_all):.4f}")
print(f"  Pearson - Std:  {np.nanstd(unrelated_pearson_all):.4f}")
print(f"  Pearson - Min:  {np.nanmin(unrelated_pearson_all):.4f}")
print(f"  Pearson - Max:  {np.nanmax(unrelated_pearson_all):.4f}")
print(f"  Spearman - Mean: {np.nanmean(unrelated_spearman_all):.4f}")
print(f"  Spearman - Std:  {np.nanstd(unrelated_spearman_all):.4f}")
print(f"  Spearman - Min:  {np.nanmin(unrelated_spearman_all):.4f}")
print(f"  Spearman - Max:  {np.nanmax(unrelated_spearman_all):.4f}")
print()

# Compute empirical p-value for the real result against the unrelated null
print("Computing empirical p-value...")
# For Pearson correlation
real_pearson = step2_results['best_avg_pearson']
null_pearson_vals = unrelated_pearson_all.flatten()
null_pearson_vals = null_pearson_vals[~np.isnan(null_pearson_vals)]  # Remove NaN

if len(null_pearson_vals) > 0:
    # p-value = fraction of null trials that meet or exceed the real result
    # For correlation, higher is more similar, so we count null >= real
    null_exceed_pearson = np.sum(null_pearson_vals >= real_pearson)
    p_value_pearson = null_exceed_pearson / len(null_pearson_vals)
else:
    p_value_pearson = 1.0

# For Spearman correlation
real_spearman = step2_results['spearman_scores'][step2_results['best_tau']]
null_spearman_vals = unrelated_spearman_all.flatten()
null_spearman_vals = null_spearman_vals[~np.isnan(null_spearman_vals)]

if len(null_spearman_vals) > 0:
    null_exceed_spearman = np.sum(null_spearman_vals >= real_spearman)
    p_value_spearman = null_exceed_spearman / len(null_spearman_vals)
else:
    p_value_spearman = 1.0

print(f"Empirical p-values (real vs. unrelated baseline):")
print(f"  Pearson: {p_value_pearson:.4f} ({null_exceed_pearson}/{len(null_pearson_vals)} null samples ≥ real)")
print(f"  Spearman: {p_value_spearman:.4f} ({null_exceed_spearman}/{len(null_spearman_vals)} null samples ≥ real)")
print()

# Interpretation
print("INTERPRETATION:")
print("- Same-distribution ceiling shows what 'definitely same origin' looks like")
print("- Unrelated floor shows what we expect from independent datasets")
print("- If real result is close to ceiling and far from floor → evidence for H1")
print("- If real result is close to floor → evidence for H0")
print("- If real result is in between → inconclusive")
print()

real_vs_ceiling = abs(step2_results['best_avg_pearson'] - best_avg_pearson_same_dist)
real_vs_floor = abs(step2_results['best_avg_pearson'] - np.nanmean(unrelated_pearson_all))

print(f"Distance from same-distribution ceiling: {real_vs_ceiling:.4f}")
print(f"Distance from unrelated floor mean: {real_vs_floor:.4f}")
print()

if real_vs_ceiling < real_vs_floor:
    print("→ Real result is closer to same-distribution ceiling (suggests H1)")
elif real_vs_ceiling > real_vs_floor:
    print("→ Real result is closer to unrelated floor (suggests H0)")
else:
    print("→ Real result is equidistant (inconclusive)")
print()

# Save results for subsequent steps
print("Step 3 complete. Results saved for subsequent steps.")
step3_results = {
    'same_dist_best_tau': best_tau_same_dist,
    'same_dist_best_pearson': best_avg_pearson_same_dist,
    'same_dist_pearson_scores': same_dist_pearson,
    'same_dist_spearman_scores': same_dist_spearman,
    'unrelated_pearson_distribution': unrelated_pearson_all,
    'unrelated_spearman_distribution': unrelated_spearman_all,
    'unrelated_pearson_mean': np.nanmean(unrelated_pearson_all),
    'unrelated_spearman_mean': np.nanmean(unrelated_spearman_all),
    'p_value_pearson': p_value_pearson,
    'p_value_spearman': p_value_spearman,
    'null_exceed_pearson': null_exceed_pearson,
    'null_total_pearson': len(null_pearson_vals),
    'null_exceed_spearman': null_exceed_spearman,
    'null_total_spearman': len(null_spearman_vals)
}

In [ ]:
# Step 4: Per-band affine fit quality at the best shift
print("=== STEP 4: PER-BAND AFFINE FIT QUALITY AT BEST SHIFT ===")
print()

# Use the best tau from Step 2
best_tau = step2_results['best_tau']
print(f"Using best shift τ* = {best_tau} from Step 2")
print()

# At τ*, for each band fit competition_curve ≈ a·train_curve + b via least squares on the 12 monthly points
print("Computing per-band affine fits (competition ≈ a × train + b)...")

affine_results = []

for b_idx, band in enumerate(bands):
    # Get the 12-point curves
    train_curve = step2_results['train_medians'][b_idx, :]  # Shape: (12,)
    test_curve = step2_results['test_medians'][b_idx, :]    # Shape: (12,)
    
    # Circularly shift competition curve by τ* months
    shifted_test_curve = np.roll(test_curve, -best_tau)
    
    # Prepare data for linear regression: y = a*x + b
    # Where y = shifted_test_curve, x = train_curve
    X = train_curve.reshape(-1, 1)  # Need 2D array for sklearn
    y = shifted_test_curve
    
    # Handle NaN values - only use points where both are not NaN
    valid_mask = ~(np.isnan(X.flatten()) | np.isnan(y))
    if np.sum(valid_mask) >= 2:  # Need at least 2 points for regression
        X_valid = X[valid_mask]
        y_valid = y[valid_mask]
        
        # Fit linear regression
        model = LinearRegression()
        model.fit(X_valid, y_valid)
        
        # Get parameters
        a = model.coef_[0]  # slope
        b = model.intercept_  # intercept
        
        # Predictions and R²
        y_pred = model.predict(X_valid)
        ss_res = np.sum((y_valid - y_pred) ** 2)
        ss_tot = np.sum((y_valid - np.mean(y_valid)) ** 2)
        if ss_tot > 0:
            r_squared = 1 - (ss_res / ss_tot)
        else:
            r_squared = 0.0 if ss_res == 0 else -np.inf
        
        # Residuals
        residuals = y_valid - y_pred
        
        affine_results.append({
            'band': band,
            'slope_a': a,
            'intercept_b': b,
            'r_squared': r_squared,
            'residuals': residuals,
            'n_points': len(X_valid),
            'train_curve': train_curve,
            'test_curve': test_curve,
            'shifted_test_curve': shifted_test_curve,
            'valid_mask': valid_mask
        })
    else:
        # Not enough valid points
        affine_results.append({
            'band': band,
            'slope_a': np.nan,
            'intercept_b': np.nan,
            'r_squared': np.nan,
            'residuals': np.array([]),
            'n_points': 0,
            'train_curve': train_curve,
            'test_curve': test_curve,
            'shifted_test_curve': shifted_test_curve,
            'valid_mask': valid_mask
        })

print("Affine fits computed.")
print()

# Create a small-multiples plot (one panel per band)
print("Creating small-multiples plot...")
n_cols = 4
n_rows = int(np.ceil(len(bands) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
elif n_cols == 1:
    axes = axes.reshape(-1, 1)

for b_idx, band in enumerate(bands):
    row = b_idx // n_cols
    col = b_idx % n_cols
    ax = axes[row, col]
    
    result = affine_results[b_idx]
    
    if result['n_points'] >= 2:
        # Plot training curve
        months_numeric = np.arange(12)
        ax.plot(months_numeric, result['train_curve'], 'o-', label='Train', color='blue', linewidth=2, markersize=4)
        
        # Plot shifted competition curve
        ax.plot(months_numeric, result['shifted_test_curve'], 's-', label=f'Test (shifted by {best_tau})', 
                color='red', linewidth=2, markersize=4)
        
        # Plot fitted line
        if not np.isnan(result['slope_a']):
            # Generate points for the fitted line
            x_fit = np.linspace(np.nanmin(result['train_curve']), np.nanmax(result['train_curve']), 100)
            y_fit = result['slope_a'] * x_fit + result['intercept_b']
            ax.plot(x_fit, y_fit, '--', label=f'Fit: y = {result["slope_a"]:.3f}x + {result["intercept_b"]:.3f}', 
                    color='green', linewidth=2)
        
        ax.set_title(f'{band} (R² = {result["r_squared"]:.3f})', fontsize=10)
        ax.set_xlabel('Month Index')
        ax.set_ylabel('Median Value')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'{band}\nInsufficient data', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{band}', fontsize=10)

# Hide empty subplots
for b_idx in range(len(bands), n_rows * n_cols):
    row = b_idx // n_cols
    col = b_idx % n_cols
    if row < n_rows and col < n_cols:
        axes[row, col].set_visible(False)

plt.suptitle(f'Per-Band Affine Fit Quality at τ* = {best_tau}\n'
             'Solid lines: actual data; Dashed line: affine fit', fontsize=16)
plt.tight_layout()
plt.show()

# Print per-band affine fit table
print("Per-band affine fit results:")
print("Band\t\tSlope (a)\tIntercept (b)\tR²\t\tN Points")
print("-" * 60)
for result in affine_results:
    if result['n_points'] >= 2:
        print(f"{result['band']:<12}\t{result['slope_a']:.4f}\t\t{result['intercept_b']:.4f}\t\t{result['r_squared']:.4f}\t\t{result['n_points']}")
    else:
        print(f"{result['band']:<12}\t{'N/A':<10}\t{'N/A':<10}\t{'N/A':<10}\t\t{result['n_points']}")
print()

# SAR-specific note: VH and VV are already in dB (log) units
print("SAR-SPECIFIC ANALYSIS:")
sar_bands = ['VH', 'VV']
for band in sar_bands:
    # Find the result for this SAR band
    sar_result = None
    for result in affine_results:
        if result['band'] == band:
            sar_result = result
            break
    
    if sar_result and sar_result['n_points'] >= 2:
        print(f"{band}:")
        print(f"  Slope (a): {sar_result['slope_a']:.4f}")
        print(f"  Intercept (b): {sar_result['intercept_b']:.4f}")
        print(f"  R²: {sar_result['r_squared']:.4f}")
        
        # Check if the fit looks affine in dB space
        # For SAR in dB: pure multiplicative rescale in linear space → additive shift in dB (a≈1, b≠0)
        # Pure additive change in linear space → non-affine in dB space (would show curved residuals)
        slope_close_to_one = abs(sar_result['slope_a'] - 1.0) < 0.1  # Within 10% of 1.0
        if slope_close_to_one:
            print(f"  → Slope close to 1.0 suggests pure multiplicative rescale in linear space")
        else:
            print(f"  → Slope significantly different from 1.0 suggests more complex transform")
        
        # Check for curved residuals (need to examine residual pattern)
        if len(sar_result['residuals']) >= 4:
            # Simple check: residuals vs predicted values correlation
            pred_vals = sar_result['slope_a'] * sar_result['train_curve'][sar_result['valid_mask']] + sar_result['intercept_b']
            if np.std(pred_vals) > 0:
                resid_corr, _ = pearsonr(pred_vals, sar_result['residuals'])
                if abs(resid_corr) > 0.3:
                    print(f"  → Notable correlation between residuals and predicted values ({resid_corr:.3f})")
                    print(f"    Suggests non-linear residual pattern - potentially non-affine in dB space")
                else:
                    print(f"  → Little correlation between residuals and predicted values ({resid_corr:.3f})")
                    print(f"    Residuals appear random - consistent with affine transform in dB space")
            else:
                print(f"  → Cannot assess residual pattern (zero variance in predictions)")
                print()
    else:
        print(f"{band}: Insufficient data for analysis")
        print()

print("Step 4 complete. Results saved for subsequent steps.")
step4_results = {
    'best_tau': best_tau,
    'affine_results': affine_results,
    'sar_analysis_complete': True
}

In [ ]:
# Step 5: Row-level correspondence search (the strongest test)
print("=== STEP 5: ROW-LEVEL CORRESPONDENCE SEARCH ===")
print()

print("This directly tests whether individual competition rows are transformed copies")
print("of individual training rows, and — unlike Steps 2-4 — doesn't require one global shift.")
print()

# For each competition row:
#   • Search over shift τ = 0..11 and over candidate training rows for the 
#     (training row, τ) pair maximizing multi-band correlation, aligned at that shift,
#     using only months both rows have in common (require ≥3 overlapping months, 
#     else skip the pair as untestable).
#   • If a full pairwise search is too slow given the row counts: reduce each row
#     to a compact shift/scale-invariant shape signature (e.g. each band z-scored,
#     then a low-dimensional summary) and use approximate nearest-neighbor search
#     to shortlist a small number of candidates per competition row, then compute
#     exact multi-band correlation only for the shortlist.
#   • State clearly which approach was used and why.

print(f"Training rows: {len(train_df_features):,}")
print(f"Competition rows: {len(test_df_features):,}")
print()

n_train = len(train_df_features)
n_test = len(test_df_features)
total_pairs = n_train * n_test

print(f"Total possible pairs: {total_pairs:,}")
print()

# Let's estimate the computational load
# For each pair, we need to check up to 12 shifts and compute correlations
# Assuming we can process ~1000 pairs per second (conservative estimate):
estimated_time_seconds = total_pairs / 1000
estimated_time_minutes = estimated_time_seconds / 60
estimated_time_hours = estimated_time_minutes / 60

print(f"Estimated processing time for full pairwise search:")
print(f"  Seconds: {estimated_time_seconds:,.0f}")
print(f"  Minutes: {estimated_time_minutes:,.1f}")
print(f"  Hours: {estimated_time_hours:,.1f}")
print()

if estimated_time_hours > 0.5:  # More than 30 minutes
    print("⚠️  WARNING: Full pairwise search may take significant time.")
    print("   Considering approximate nearest-neighbor approach for efficiency.")
    use_approximate = True
else:
    print("✅ Full pairwise search should be feasible.")
    use_approximate = False
print()

# Pre-compute which months are observed for each row to speed up overlap checking
def get_observed_months_mask(df_features, bands, months):
    """Return a boolean matrix indicating which months are observed for each row"""
    n_rows = len(df_features)
    n_months = len(months)
    # Shape: (n_rows, n_months) - True if month is observed (at least one band has data)
    observed_mask = np.zeros((n_rows, n_months), dtype=bool)

    for row_idx in range(n_rows):
        for month_idx, month in enumerate(months):
            # Check if ANY band for this month has data
            month_has_data = False
            for band in bands:
                feature = f"{band}_{month}"
                val = df_features.iloc[row_idx][feature]
                if not pd.isna(val):
                    month_has_data = True
                    break
            observed_mask[row_idx, month_idx] = month_has_data

    return observed_mask

print("Pre-computing observed months masks...")
train_observed_mask = get_observed_months_mask(train_df_features, bands, months)
test_observed_mask = get_observed_months_mask(test_df_features, bands, months)
print("Observed months masks computed.")
print()

# Function to count overlapping months between two rows (considering shift)
def count_overlapping_months(train_row_idx, test_row_idx, shift, train_mask, test_mask):
    """Count months where both rows have data after applying shift to test row"""
    # Shift the test mask: shifted_test_mask[m] = test_mask[(m - shift) mod 12]
    # This means: what month m in train corresponds to month (m - shift) in original test
    shifted_test_mask = np.roll(test_mask[test_row_idx], -shift)
    # Count positions where both are True
    overlap = np.logical_and(train_mask[train_row_idx], shifted_test_mask)
    return np.sum(overlap)

# Function to compute multi-band correlation for a pair of rows at a given shift
def compute_multiband_correlation(train_row_idx, test_row_idx, shift, train_features, test_features, bands, months, min_overlap=3):
    """Compute correlation between train and shifted test rows using overlapping months"""
    # First check if we have sufficient overlap
    overlap_count = count_overlapping_months(train_row_idx, test_row_idx, shift,
                                           train_observed_mask, test_observed_mask)
    if overlap_count < min_overlap:
        return -np.inf, []  # Return negative infinity and empty details

    # Collect all valid (train_value, test_value) pairs across bands and overlapping months
    train_values = []
    test_values = []
    details = []  # Store (band, month, train_val, test_val) for debugging

    for band_idx, band in enumerate(bands):
        for month_idx, month in enumerate(months):
            # Check if this month is observable in both rows (after shift)
            train_has_data = train_observed_mask[train_row_idx, month_idx]
            # For test, we need to check the original month that maps to this train month after shift
            original_test_month_idx = (month_idx - shift) % 12
            test_has_data = test_observed_mask[test_row_idx, original_test_month_idx]

            if train_has_data and test_has_data:
                # Get the actual values
                train_feature = f"{band}_{month}"
                test_feature = f"{band}_{months[original_test_month_idx]}"  # Convert back to actual month name

                train_val = train_features.iloc[train_row_idx][train_feature]
                test_val = test_features.iloc[test_row_idx][test_feature]

                if not pd.isna(train_val) and not pd.isna(test_val):
                    train_values.append(train_val)
                    test_values.append(test_val)
                    details.append((band, month, train_val, test_val))

    # Need at least 2 points for correlation
    if len(train_values) < 2:
        return -np.inf, details

    # Compute Pearson correlation across all bands and months
    try:
        correlation, _ = stats.pearsonr(train_values, test_values)
        return correlation, details
    except:
        return -np.inf, details

# Now perform the search - EITHER full search OR approximate based on flag
print("Starting search...")
print()

# Store results for each test row
best_matches = []  # List of dicts with best match info for each test row

if use_approximate:
    print("Using APPROXIMATE nearest-neighbor approach for efficiency")
    print("Step 1: Computing shift/scale-invariant signatures for all rows")

    # Function to compute shift/scale-invariant signature for a row
    def compute_signature(row_features, bands, months):
        """Compute a signature that is invariant to circular shifts and affine transforms"""
        # For each band, compute the normalized pattern across months
        signatures = []

        for band in bands:
            # Extract values for this band across all months
            band_vals = []
            valid_months = []

            for month_idx, month in enumerate(months):
                feature = f"{band}_{month}"
                val = row_features[feature]
                if not pd.isna(val):
                    band_vals.append(val)
                    valid_months.append(month_idx)

            if len(band_vals) >= 2:
                # Normalize to zero mean, unit variance (z-score)
                band_vals = np.array(band_vals)
                if np.std(band_vals) > 0:
                    band_vals = (band_vals - np.mean(band_vals)) / np.std(band_vals)
                else:
                    band_vals = np.zeros_like(band_vals)

                # Create a 12-dimensional vector with zeros for missing months
                full_vector = np.zeros(12)
                for i, month_idx in enumerate(valid_months):
                    full_vector[month_idx] = band_vals[i]

                signatures.append(full_vector)
            else:
                # Not enough data for this band - use zeros
                signatures.append(np.zeros(12))

        # Concatenate all band signatures
        return np.concatenate(signatures)

    # Compute signatures for training rows
    print("Computing training signatures...")
    train_signatures = []
    for train_idx in range(n_train):
        signature = compute_signature(train_df_features.iloc[train_idx], bands, months)
        train_signatures.append(signature)
        if train_idx % 1000 == 0 and train_idx > 0:
            print(f"  Computed {train_idx}/{n_train} training signatures")
    train_signatures = np.array(train_signatures)
    print("Training signatures computed.")

    # Compute signatures for test rows
    print("Computing test signatures...")
    test_signatures = []
    for test_idx in range(n_test):
        signature = compute_signature(test_df_features.iloc[test_idx], bands, months)
        test_signatures.append(signature)
        if test_idx % 1000 == 0 and test_idx > 0:
            print(f"  Computed {test_idx}/{n_test} test signatures")
    test_signatures = np.array(test_signatures)
    print("Test signatures computed.")

    # Build approximate nearest neighbor index (using brute force for simplicity,
    # but could use sklearn.neighbors.NearestNeighbors for large datasets)
    print("Building approximate nearest neighbor index...")
    # For now, we'll use a simplified approach: for each test row,
    # find top-K similar training rows using signature distance

    K = min(50, max(1, n_train // 20))  # Number of candidates to consider per test row
    if K < 1:
        K = 1

    print(f"Using top {K} candidates per test row based on signature similarity")

    for test_idx in range(n_test):
        if test_idx % 100 == 0:
            print(f"  Processing test row {test_idx}/{n_test} ({test_idx/n_test*100:.1f}%)")

        # Compute signature distances to all training rows
        test_sig = test_signatures[test_idx]
        # Use Euclidean distance for signature similarity
        distances = np.linalg.norm(train_signatures - test_sig, axis=1)

        # Get indices of K nearest neighbors
        nearest_indices = np.argsort(distances)[:K]

        # Now do exact search only on these candidates
        best_correlation = -np.inf
        best_train_idx = -1
        best_shift = -1
        best_details = []

        # Search over candidate training rows and all shifts
        for train_idx in nearest_indices:
            for shift in range(12):
                correlation, details = compute_multiband_correlation(
                    train_idx, test_idx, shift,
                    train_df_features, test_df_features, bands, months
                )

                if correlation > best_correlation:
                    best_correlation = correlation
                    best_train_idx = train_idx
                    best_shift = shift
                    best_details = details

        # Store the best match for this test row
        best_matches.append({
            'test_row_idx': test_idx,
            'test_row_id': test_ids.iloc[test_idx],
            'best_train_idx': best_train_idx,
            'best_train_id': train_ids.iloc[best_train_idx] if best_train_idx >= 0 else None,
            'best_shift': best_shift,
            'best_correlation': best_correlation,
            'n_matching_points': len(best_details),
            'match_details': best_details[:5] if len(best_details) > 5 else best_details  # Store first 5 details
        })

else:
    print("Using FULL pairwise search with optimizations")

    for test_idx in range(n_test):
        if test_idx % 100 == 0:
            print(f"  Processing test row {test_idx}/{n_test} ({test_idx/n_test*100:.1f}%)")

        best_correlation = -np.inf
        best_train_idx = -1
        best_shift = -1
        best_details = []

        # Search over all training rows and shifts
        # Optimization: break early if we find a perfect correlation (1.0)
        perfect_found = False

        for train_idx in range(n_train):
            # Early break if perfect match found
            if perfect_found:
                break

            for shift in range(12):
                correlation, details = compute_multiband_correlation(
                    train_idx, test_idx, shift,
                    train_df_features, test_df_features, bands, months
                )

                if correlation > best_correlation:
                    best_correlation = correlation
                    best_train_idx = train_idx
                    best_shift = shift
                    best_details = details

                    # Early break if we find a perfect correlation
                    if correlation >= 0.999:  # Nearly perfect
                        perfect_found = True
                        break

            # Additional optimization: if we've checked several shifts and found
            # a very good correlation, we might skip remaining shifts for this train row
            # (This is a simplified version - in practice could be more sophisticated)

        # Store the best match for this test row
        best_matches.append({
            'test_row_idx': test_idx,
            'test_row_id': test_ids.iloc[test_idx],
            'best_train_idx': best_train_idx,
            'best_train_id': train_ids.iloc[best_train_idx] if best_train_idx >= 0 else None,
            'best_shift': best_shift,
            'best_correlation': best_correlation,
            'n_matching_points': len(best_details),
            'match_details': best_details[:5] if len(best_details) > 5 else best_details  # Store first 5 details
        })

print(f"Search completed. Processed {n_test} test rows.")
print()

# Extract the best correlations for analysis
correlations = [match['best_correlation'] for match in best_matches if match['best_correlation'] > -np.inf]
valid_matches = [match for match in best_matches if match['best_correlation'] > -np.inf]

print(f"Number of test rows with valid matches (≥2 overlapping points): {len(valid_matches)}/{n_test}")
print()

if len(correlations) > 0:
    correlations_array = np.array(correlations)
    print(f"Best match correlation statistics:")
    print(f"  Mean: {np.mean(correlations_array):.4f}")
    print(f"  Median: {np.median(correlations_array):.4f}")
    print(f"  Std: {np.std(correlations_array):.4f}")
    print(f"  Min: {np.min(correlations_array):.4f}")
    print(f"  Max: {np.max(correlations_array):.4f}")
    print()

    # Show top matches
    print("Top 10 best matches (highest correlation):")
    sorted_matches = sorted(valid_matches, key=lambda x: x['best_correlation'], reverse=True)
    print("Rank\tTest ID\t\t\tTrain ID\t\tShift\tCorr\t#Points")
    print("-" * 80)
    for i, match in enumerate(sorted_matches[:10]):
        print(f"{i+1:2d}\t{match['test_row_id']}\t{match['best_train_id']}\t{match['best_shift']:2d}\t{match['best_correlation']:.4f}\t{match['n_matching_points']}")
    print()
else:
    print("No valid matches found!")
    print()

# Build the correct null distribution — not a naive one:
print("Building null distribution using shuffled competition set...")
print("(This ensures we're comparing 'best-of-N under real data' against 'best-of-N under known-unrelated data.')")

# Create shuffled test set (same as in Step 3)
np.random.seed(42)  # For reproducibility
shuffled_test_indices = np.random.permutation(n_test)
shuffled_test_df = test_df_features.iloc[shuffled_test_indices]
shuffled_test_ids = test_ids.iloc[shuffled_test_indices]

print("Searching for best matches in shuffled test set...")
shuffled_best_matches = []

for test_idx in range(n_test):
    if test_idx % 100 == 0:
        print(f"  Processing shuffled test row {test_idx}/{n_test} ({test_idx/n_test*100:.1f}%)")

    best_correlation = -np.inf
    best_train_idx = -1
    best_shift = -1
    best_details = []

    # Search over all training rows and shifts
    for train_idx in range(n_train):
        for shift in range(12):
            correlation, details = compute_multiband_correlation(
                train_idx, test_idx, shift,
                train_df_features, shuffled_test_df, bands, months
            )

            if correlation > best_correlation:
                best_correlation = correlation
                best_train_idx = train_idx
                best_shift = shift
                best_details = details

    # Store the best match for this shuffled test row
    shuffled_best_matches.append({
        'shuffled_test_row_idx': test_idx,
        'original_test_row_idx': shuffled_test_indices[test_idx],
        'shuffled_test_id': shuffled_test_ids[test_idx],
        'best_train_idx': best_train_idx,
        'best_train_id': train_ids.iloc[best_train_idx] if best_train_idx >= 0 else None,
        'best_shift': best_shift,
        'best_correlation': best_correlation,
        'n_matching_points': len(best_details)
    })

print("Null distribution search completed.")
print()

# Extract null correlations
null_correlations = [match['best_correlation'] for match in shuffled_best_matches if match['best_correlation'] > -np.inf]
print(f"Number of shuffled test rows with valid matches: {len(null_correlations)}/{n_test}")
print()

if len(null_correlations) > 0:
    null_correlations_array = np.array(null_correlations)
    print(f"Null distribution correlation statistics:")
    print(f"  Mean: {np.mean(null_correlations_array):.4f}")
    print(f"  Median: {np.median(null_correlations_array):.4f}")
    print(f"  Std: {np.std(null_correlations_array):.4f}")
    print(f"  Min: {np.min(null_correlations_array):.4f}")
    print(f"  Max: {np.max(null_correlations_array):.4f}")
    print()

    # Compute empirical p-value for the best real matches
    if len(correlations) > 0:
        # For the real data, let's look at the top correlations
        top_real_correlation = np.max(correlations_array) if len(correlations_array) > 0 else -np.inf
        # p-value = fraction of null trials that meet or exceed this top real correlation
        null_exceed_count = np.sum(null_correlations_array >= top_real_correlation)
        empirical_p_value_top = null_exceed_count / len(null_correlations_array) if len(null_correlations_array) > 0 else 1.0

        print(f"Top real correlation: {top_real_correlation:.4f}")
        print(f"Empirical p-value (top real correlation vs null): {empirical_p_value_top:.4f}")
        print(f"Number of null samples ≥ top real: {null_exceed_count} out of {len(null_correlations_array)}")
        print()
        
        # Also compute for median correlation
        median_real_correlation = np.median(correlations_array)
        null_exceed_median = np.sum(null_correlations_array >= median_real_correlation)
        empirical_p_value_median = null_exceed_median / len(null_correlations_array) if len(null_correlations_array) > 0 else 1.0
        
        print(f"Median real correlation: {median_real_correlation:.4f}")
        print(f"Empirical p-value (median real correlation vs null): {empirical_p_value_median:.4f}")
        print(f"Number of null samples ≥ median real: {null_exceed_median} out of {len(null_correlations_array)}")
        print()
else:
    print("Could not build null distribution - no valid matches in shuffled data.")
    print()

# For the handful of best matches found:
print("Analyzing top matches...")
if len(sorted_matches) > 0:
    top_n = min(5, len(sorted_matches))
    print(f"Examining top {top_n} matches in detail:")
    
    for i in range(top_n):
        match = sorted_matches[i]
        print(f"\n--- Match #{i+1} ---")
        print(f"Test ID: {match['test_row_id']}")
        print(f"Train ID: {match['best_train_id']}")
        print(f"Shift: {match['best_shift']}")
        print(f"Correlation: {match['best_correlation']:.4f}")
        print(f"Matching points: {match['n_matching_points']}")
        
        # Scatter-plot raw values (train vs. matched competition, per band)
        if match['best_train_idx'] >= 0 and match['n_matching_points'] >= 2:
            print("Generating scatter plots for each band...")
            
            # Create subplots for bands
            n_sar_bands = 2  # VH, VV
            n_optical_bands = len([b for b in bands if b not in ['VH', 'VV']])
            n_cols = 4
            n_rows = int(np.ceil(len(bands) / n_cols))
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
            if n_rows == 1:
                axes = axes.reshape(1, -1)
            elif n_cols == 1:
                axes = axes.reshape(-1, 1)
            
            train_row = train_df_features.iloc[match['best_train_idx']]
            test_row = test_df_features.iloc[match['test_row_idx']]
            shift = match['best_shift']
            
            plot_idx = 0
            for band_idx, band in enumerate(bands):
                row = plot_idx // n_cols
                col = plot_idx % n_cols
                ax = axes[row, col]
                
                # Collect all matched points for this band
                band_train_vals = []
                band_test_vals = []
                
                for detail in match['match_details']:
                    if detail[0] == band:  # band match
                        band_train_vals.append(detail[2])  # train_val
                        band_test_vals.append(detail[3])   # test_val
                
                # Also check all months for additional points
                for month_idx, month in enumerate(months):
                    # Check if this month is observable in both rows (after shift)
                    train_has_data = train_observed_mask[match['best_train_idx'], month_idx]
                    original_test_month_idx = (month_idx - shift) % 12
                    test_has_data = test_observed_mask[match['test_row_idx'], original_test_month_idx]
                    
                    if train_has_data and test_has_data:
                        train_feature = f"{band}_{month}"
                        test_feature = f"{band}_{months[original_test_month_idx]}"
                        
                        train_val = train_row[train_feature]
                        test_val = test_row[test_feature]
                        
                        if not pd.isna(train_val) and not pd.isna(test_val):
                            # Avoid duplicates from match_details
                            duplicate = False
                            for existing_train, existing_test in zip(band_train_vals, band_test_vals):
                                if abs(existing_train - train_val) < 1e-6 and abs(existing_test - test_val) < 1e-6:
                                    duplicate = True
                                    break
                            if not duplicate:
                                band_train_vals.append(train_val)
                                band_test_vals.append(test_val)
                
                if len(band_train_vals) >= 2:
                    ax.scatter(band_train_vals, band_test_vals, alpha=0.7, s=30)
                    
                    # Add perfect correlation line (y = x)
                    min_val = min(min(band_train_vals), min(band_test_vals))
                    max_val = max(max(band_train_vals), max(band_test_vals))
                    ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label='Perfect match (y=x)')
                    
                    # Compute and show R²
                    if len(band_train_vals) > 1 and np.std(band_train_vals) > 0 and np.std(band_test_vals) > 0:
                        corr, _ = pearsonr(band_train_vals, band_test_vals)
                        r_squared = corr ** 2
                        ax.text(0.05, 0.95, f'R² = {r_squared:.3f}', 
                                transform=ax.transAxes, fontsize=9, 
                                verticalalignment='top',
                                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
                    
                    ax.set_xlabel('Train Value')
                    ax.set_ylabel('Test Value')
                    ax.set_title(f'{band}')
                    ax.grid(True, alpha=0.3)
                    ax.legend(fontsize=8)
                else:
                    ax.text(0.5, 0.5, f'Insufficient data\nfor {band}', 
                            ha='center', va='center', transform=ax.transAxes)
                    ax.set_title(f'{band}')
                
                plot_idx += 1
            
            # Hide empty subplots
            for idx in range(len(bands), n_rows * n_cols):
                row = idx // n_cols
                col = idx % n_cols
                if row < n_rows and col < n_cols:
                    axes[row, col].set_visible(False)
            
            plt.suptitle(f'Match #{i+1}: Train ID {match["best_train_id"]} vs Test ID {match["test_row_id"]}\n'
                         f'Shift = {match["best_shift"]}, Correlation = {match["best_correlation"]:.4f}', 
                         fontsize=14)
            plt.tight_layout()
            plt.show()
            
            # Check correlation of the residuals after removing the smooth 12-point trend
            print(f"  Checking residual correlations after detrending...")
            if len(band_train_vals) >= 3 and len(band_test_vals) >= 3:
                # For each band, detrend and compute correlation of residuals
                all_train_residuals = []
                all_test_residuals = []
                
                for band in bands:
                    # Get all matched points for this band
                    band_train_vals = []
                    band_test_vals = []
                    
                    for detail in match['match_details']:
                        if detail[0] == band:
                            band_train_vals.append(detail[2])
                            band_test_vals.append(detail[3])
                    
                    # Check all months
                    for month_idx, month in enumerate(months):
                        train_has_data = train_observed_mask[match['best_train_idx'], month_idx]
                        original_test_month_idx = (month_idx - shift) % 12
                        test_has_data = test_observed_mask[match['test_row_idx'], original_test_month_idx]
                        
                        if train_has_data and test_has_data:
                            train_feature = f"{band}_{month}"
                            test_feature = f"{band}_{months[original_test_month_idx]}"
                            
                            train_val = train_row[train_feature]
                            test_val = test_row[test_feature]
                            
                            if not pd.isna(train_val) and not pd.isna(test_val):
                                duplicate = False
                                for existing_train, existing_test in zip(band_train_vals, band_test_vals):
                                    if abs(existing_train - train_val) < 1e-6 and abs(existing_test - test_val) < 1e-6:
                                        duplicate = True
                                        break
                                if not duplicate:
                                    band_train_vals.append(train_val)
                                    band_test_vals.append(test_val)
                    
                    if len(band_train_vals) >= 2:
                        # Detrend by subtracting median (simple approach)
                        train_median = np.median(band_train_vals)
                        test_median = np.median(band_test_vals)
                        
                        train_detrended = np.array(band_train_vals) - train_median
                        test_detrended = np.array(band_test_vals) - test_median
                        
                        all_train_residuals.extend(train_detrended)
                        all_test_residuals.extend(test_detrended)
                
                if len(all_train_residuals) >= 2 and len(all_test_residuals) >= 2:
                    if np.std(all_train_residuals) > 0 and np.std(all_test_residuals) > 0:
                        resid_corr, _ = pearsonr(all_train_residuals, all_test_residuals)
                        print(f"    Correlation of detrended residuals: {resid_corr:.4f}")
                        if abs(resid_corr) > 0.5:
                            print(f"    → Strong residual correlation suggests fine-grained similarities")
                        elif abs(resid_corr) > 0.3:
                            print(f"    → Moderate residual correlation")
                        else:
                            print(f"    → Weak residual correlation")
                    else:
                        print(f"    → Cannot compute residual correlation (zero variance)")
                else:
                    print(f"    → Insufficient data for residual analysis")
            else:
                print(f"    → Insufficient data points for residual analysis")
        else:
            print("  Insufficient data for detailed analysis")
    
    print("\nStep 5 complete.")
else:
    print("No valid matches to analyze.")
    print("Step 5 complete.")

# Save results for subsequent steps
print("Step 5 complete. Results saved for subsequent steps.")
step5_results = {
    'use_approximate': use_approximate,
    'best_matches': best_matches,
    'valid_matches': valid_matches,
    'correlations_array': np.array(correlations) if len(correlations) > 0 else np.array([]),
    'shuffled_best_matches': shuffled_best_matches,
    'null_correlations_array': np.array(null_correlations) if len(null_correlations) > 0 else np.array([]),
    'top_matches': sorted_matches[:10] if len(sorted_matches) > 0 else []
}

In [ ]:
# Step 6: Missingness-pattern fingerprint (values-independent check)
print("=== STEP 6: MISSINGNESS-PATTERN FINGERPRINT ===")
print()

print("Represent each row's 'which months are observed' as a 12-bit pattern.")
print("Independent of the actual band values: check whether competition rows' patterns,")
print("shifted by τ* (or searched per-row), match training rows' patterns more often than")
print("a random-permutation baseline would predict.")
print()

# Use the best tau from Step 2 as the primary shift to test
best_tau = step2_results['best_tau']
print(f"Using best shift τ* = {best_tau} from Step 2 as primary hypothesis")
print()

# Create 12-bit patterns for each row (1 = observed, 0 = missing)
def create_missingness_patterns(df_features, bands, months):
    """Create binary missingness patterns for each row"""
    n_rows = len(df_features)
    patterns = np.zeros((n_rows, 12), dtype=int)  # 12 months
    
    for row_idx in range(n_rows):
        for month_idx, month in enumerate(months):
            # Check if ANY band for this month has data (not NaN and not -9999)
            month_has_data = False
            for band in bands:
                feature = f"{band}_{month}"
                val = df_features.iloc[row_idx][feature]
                if not pd.isna(val):  # We already converted -9999 to NaN
                    month_has_data = True
                    break
            patterns[row_idx, month_idx] = 1 if month_has_data else 0
    
    return patterns

print("Creating missingness patterns...")
train_patterns = create_missingness_patterns(train_df_features, bands, months)
test_patterns = create_missingness_patterns(test_df_features, bands, months)
print(f"Training patterns shape: {train_patterns.shape}")
print(f"Test patterns shape: {test_patterns.shape}")
print()

# Function to compute match rate between two sets of patterns with a given shift
def compute_pattern_match_rate(patterns1, patterns2, shift):
    """Compute fraction of rows in patterns2 that have a matching pattern in patterns1 when shifted"""
    n2 = len(patterns2)
    matches = 0
    
    for i in range(n2):
        # Shift patterns2[i] by shift positions
        shifted_pattern = np.roll(patterns2[i], -shift)
        # Check if this shifted pattern exists exactly in patterns1
        for j in range(len(patterns1)):
            if np.array_equal(shifted_pattern, patterns1[j]):
                matches += 1
                break  # Found a match, move to next test row
    
    return matches / n2 if n2 > 0 else 0

# Test the hypothesis: competition patterns shifted by τ* match training patterns
print(f"Testing: Competition patterns shifted by τ*={best_tau} vs Training patterns")
actual_match_rate = compute_pattern_match_rate(train_patterns, test_patterns, best_tau)
print(f"Actual match rate: {actual_match_rate:.4f} ({actual_match_rate*100:.1f}%)")
print()

# Also test per-row shift search (more flexible, like in Step 5)
print("Testing: Per-row optimal shift search (like Step 5)")
per_row_match_count = 0
for test_idx in len(test_patterns)):
    best_match_for_this_test = False
    # Try all possible shifts
    for shift in range(12):
        shifted_pattern = np.roll(test_patterns[test_idx], -shift)
        # Check if this shifted pattern exists in training patterns
        for train_idx in range(len(train_patterns)):
            if np.array_equal(shifted_pattern, train_patterns[train_idx]):
                best_match_for_this_test = True
                break
        if best_match_for_this_test:
            break
    if best_match_for_this_test:
        per_row_match_count += 1

per_row_match_rate = per_row_match_count / len(test_patterns) if len(test_patterns) > 0 else 0
print(f"Per-row optimal shift match rate: {per_row_match_rate:.4f} ({per_row_match_rate*100:.1f}%)")
print()

# Build random-permutation baseline
print("Building random-permutation baseline...")
print("Shuffling training patterns to destroy any real correspondence while preserving")
print("the distribution of pattern types.")
print()

n_permutations = 20
baseline_match_rates = []
baseline_per_row_match_rates = []

for perm_idx in range(n_permutations):
    if perm_idx % 5 == 0:
        print(f"  Permutation {perm_idx+1}/{n_permutations}")
    
    # Shuffle the training patterns (destroy row-wise correspondence)
    shuffled_indices = np.random.permutation(len(train_patterns))
    shuffled_train_patterns = train_patterns[shuffled_indices]
    
    # Compute match rate with fixed shift
    fixed_shift_rate = compute_pattern_match_rate(shuffled_train_patterns, test_patterns, best_tau)
    baseline_match_rates.append(fixed_shift_rate)
    
    # Compute per-row optimal shift rate
    per_row_matches = 0
    for test_idx in range(len(test_patterns)):
        best_match_for_this_test = False
        for shift in range(12):
            shifted_pattern = np.roll(test_patterns[test_idx], -shift)
            for train_idx in range(len(shuffled_train_patterns)):
                if np.array_equal(shifted_pattern, shuffled_train_patterns[train_idx]):
                    best_match_for_this_test = True
                    break
            if best_match_for_this_test:
                break
        if best_match_for_this_test:
            per_row_matches += 1
    
    per_row_rate = per_row_matches / len(test_patterns) if len(test_patterns) > 0 else 0
    baseline_per_row_match_rates.append(per_row_rate)

baseline_match_rates = np.array(baseline_match_rates)
baseline_per_row_match_rates = np.array(baseline_per_row_match_rates)

print("Baseline computation complete.")
print()

print("=== BASELINE COMPARISON ===")
print(f"Fixed shift τ*={best_tau}:")
print(f"  Actual match rate: {actual_match_rate:.4f}")
print(f"  Baseline - Mean: {np.mean(baseline_match_rates):.4f}")
print(f"  Baseline - Std:  {np.std(baseline_match_rates):.4f}")
print(f"  Baseline - Min:  {np.min(baseline_match_rates):.4f}")
print(f"  Baseline - Max:  {np.max(baseline_match_rates):.4f}")
print()

# Compute empirical p-value
null_exceed_count = np.sum(baseline_match_rates >= actual_match_rate)
empirical_p_value = null_exceed_count / len(baseline_match_rates) if len(baseline_match_rates) > 0 else 1.0
print(f"Empirical p-value (fixed shift): {empirical_p_value:.4f}")
print(f"  ({null_exceed_count}/{len(baseline_match_rates)} baseline ≥ actual)")
print()

print(f"Per-row optimal shift:")
print(f"  Actual match rate: {per_row_match_rate:.4f}")
print(f"  Baseline - Mean: {np.mean(baseline_per_row_match_rates):.4f}")
print(f"  Baseline - Std:  {np.std(baseline_per_row_match_rates):.4f}")
print(f"  Baseline - Min:  {np.min(baseline_per_row_match_rates):.4f}")
print(f"  Baseline - Max:  {np.max(baseline_per_row_match_rates):.4f}")
print()

null_exceed_per_row = np.sum(baseline_per_row_match_rates >= per_row_match_rate)
empirical_p_value_per_row = null_exceed_per_row / len(baseline_per_row_match_rates) if len(baseline_per_row_match_rates) > 0 else 1.0
print(f"Empirical p-value (per-row shift): {empirical_p_value_per_row:.4f}")
print(f"  ({null_exceed_per_row}/{len(baseline_per_row_match_rates)} baseline ≥ actual)")
print()

# Interpretation
print("INTERPRETATION:")
print("- Low p-value (< 0.05) suggests the observed match rate is unlikely by chance")
print("- High match rate with low p-value supports H1 (same origin)")
print("- High match rate with high p-value could still be coincidental")
print("- Low match rate suggests different missingness patterns (supports H0)")
print()

if empirical_p_value < 0.05:
    print(f"→ Fixed shift test: SIGNIFICANT (p = {empirical_p_value:.4f})")
    print("   Competition missingness patterns match training patterns when shifted by τ*")
    print("   more often than expected by chance → Supports H1")
elif empirical_p_value < 0.10:
    print(f"→ Fixed shift test: SUGGESTIVE (p = {empirical_p_value:.4f})")
    print("   Some evidence for non-random matching")
else:
    print(f"→ Fixed shift test: NOT SIGNIFICANT (p = {empirical_p_value:.4f})")
    print("   Observed matching consistent with random chance")

print()
if empirical_p_value_per_row < 0.05:
    print(f"→ Per-row shift test: SIGNIFICANT (p = {empirical_p_value_per_row:.4f})")
    print("   Competition missingness patterns match training patterns with some shift")
    print("   more often than expected by chance → Supports H1")
elif empirical_p_value_per_row < 0.10:
    print(f"→ Per-row shift test: SUGGESTIVE (p = {empirical_p_value_per_row:.4f})")
else:
    print(f"→ Per-row shift test: NOT SIGNIFICANT (p = {empirical_p_value_per_row:.4f})")
print()

# Additional analysis: show most common patterns
print("=== PATTERN ANALYSIS ===")
print("Most common missingness patterns in training:")

# Count pattern frequencies
from collections import Counter
train_pattern_tuples = [tuple(row) for row in train_patterns]
test_pattern_tuples = [tuple(row) for row in test_patterns]

train_counter = Counter(train_pattern_tuples)
test_counter = Counter(test_pattern_tuples)

print("Top 5 training patterns:")
for pattern, count in train_counter.most_common(5):
    percentage = count / len(train_patterns) * 100
    # Convert to readable format
    pattern_str = ''.join(['1' if bit else '0' for bit in pattern])
    observed_months = [str(i+1) for i, bit in enumerate(pattern) if bit == 1]
    print(f"  {pattern_str} ({count:4d} rows, {percentage:5.1f}%): Months {','.join(observed_months) if observed_months else 'None'}")

print()
print("Top 5 test patterns:")
for pattern, count in test_counter.most_common(5):
    percentage = count / len(test_patterns) * 100
    pattern_str = ''.join(['1' if bit else '0' for bit in pattern])
    observed_months = [str(i+1) for i, bit in enumerate(pattern) if bit == 1]
    print(f"  {pattern_str} ({count:4d} rows, {percentage:5.1f}%): Months {','.join(observed_months) if observed_months else 'None'}")

print()
print("Step 6 complete. Results saved for subsequent steps.")
step6_results = {
    'best_tau': best_tau,
    'actual_match_rate': actual_match_rate,
    'per_row_match_rate': per_row_match_rate,
    'baseline_match_rates': baseline_match_rates,
    'baseline_per_row_match_rates': baseline_per_row_match_rates,
    'empirical_p_value': empirical_p_value,
    'empirical_p_value_per_row': empirical_p_value_per_row,
    'train_patterns': train_patterns,
    'test_patterns': test_patterns,
    'train_pattern_counter': train_counter,
    'test_pattern_counter': test_counter
}

In [ ]:
# Step 7: Inter-band correlation structure preservation
print("=== STEP 7: INTER-BAND CORRELATION STRUCTURE PRESERVATION ===")
print()

print("Compute the 12x12 band-to-band correlation matrix (across rows, using available data)")
print("separately for train and competition.")
print("A per-band affine transform preserves Pearson correlation between bands exactly")
print("— so if H1 is true, these matrices should closely match.")
print()

# Function to compute band-band correlation matrix
def compute_band_band_correlation(df, bands):
    """Compute 12x12 correlation matrix between bands"""
    n_bands = len(bands)
    corr_matrix = np.full((n_bands, n_bands), np.nan)
    
    for i, band_i in enumerate(bands):
        for j, band_j in enumerate(bands):
            if i == j:
                corr_matrix[i, j] = 1.0  # Correlation of band with itself
            else:
                # Get data for both bands
                feature_i = f"{band_i}_{{month}}"  # We'll handle months below
                feature_j = f"{band_j}_{{month}}"
                
                # Collect all valid pairs across all months
                band_i_values = []
                band_j_values = []
                
                # Check each month
                for month in [f"{m:02d}" for m in range(1, 13)]:
                    fi = f"{band_i}_{month}"
                    fj = f"{band_j}_{month}"
                    
                    if fi in df.columns and fj in df.columns:
                        # Get valid pairs (both not NaN)
                        vals_i = df[fi].dropna()
                        vals_j = df[fj].dropna()
                        
                        # Find common indices
                        common_idx = vals_i.index.intersection(vals_j.index)
                        if len(common_idx) >= 2:  # Need at least 2 points
                            band_i_values.extend(df.loc[common_idx, fi].values)
                            band_j_values.extend(df.loc[common_idx, fj].values)
                
                # Compute correlation if we have enough data
                if len(band_i_values) >= 2 and len(band_j_values) >= 2:
                    # Check for sufficient variation
                    if np.std(band_i_values) > 0 and np.std(band_j_values) > 0:
                        try:
                            corr, _ = pearsonr(band_i_values, band_j_values)
                            corr_matrix[i, j] = corr
                        except:
                            corr_matrix[i, j] = np.nan
                    else:
                        corr_matrix[i, j] = np.nan  # No variation in one or both
                else:
                    corr_matrix[i, j] = np.nan  # Insufficient data
    
    return corr_matrix

print("Computing training band-band correlation matrix...")
train_corr_matrix = compute_band_band_correlation(train_df_features, bands)
print("Computing competition band-band correlation matrix...")
test_corr_matrix = compute_band_band_correlation(test_df_features, bands)

print("Correlation matrices computed.")
print()

# Handle NaN values for comparison (replace with 0 for difference calculation, but keep NaN for display)
train_corr_clean = np.nan_to_num(train_corr_matrix, nan=0.0)
test_corr_clean = np.nan_to_num(test_corr_matrix, nan=0.0)

# Compute difference matrix
diff_matrix = train_corr_clean - test_corr_clean

# Scalar summary: correlation between the two flattened matrices
# Only use elements where both are not NaN
valid_elements = ~(np.isnan(train_corr_matrix) | np.isnan(test_corr_matrix))
if np.sum(valid_elements) > 0:
    train_flat = train_corr_matrix[valid_elements]
    test_flat = test_corr_matrix[valid_elements]
    
    if np.std(train_flat) > 0 and np.std(test_flat) > 0:
        matrix_correlation, _ = pearsonr(train_flat, test_flat)
        frobenius_norm = np.sqrt(np.nansum(diff_matrix ** 2))
        mean_absolute_error = np.nanmean(np.abs(diff_matrix))
    else:
        matrix_correlation = np.nan
        frobenius_norm = np.nan
        mean_absolute_error = np.nan
else:
    matrix_correlation = np.nan
    frobenius_norm = np.nan
    mean_absolute_error = np.nan

print("=== RESULTS ===")
print(f"Matrix correlation (Pearson between flattened matrices): {matrix_correlation:.4f}")
print(f"Frobenius norm of difference: {frobenius_norm:.4f}")
print(f"Mean absolute error: {mean_absolute_error:.4f}")
print()

# Create visualizations
print("Creating visualizations...")

# Set up the plotting area
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Training correlation matrix
im1 = axes[0, 0].imshow(train_corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0, 0].set_title('Training Band-Band Correlation Matrix')
axes[0, 0].set_xticks(range(len(bands)))
axes[0, 0].set_xticklabels(bands, rotation=45, ha='right')
axes[0, 0].set_yticks(range(len(bands)))
axes[0, 0].set_yticklabels(bands)
plt.colorbar(im1, ax=axes[0, 0])

# Competition correlation matrix
im2 = axes[0, 1].imshow(test_corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0, 1].set_title('Competition Band-Band Correlation Matrix')
axes[0, 1].set_xticks(range(len(bands)))
axes[0, 1].set_xticklabels(bands, rotation=45, ha='right')
axes[0, 1].set_yticks(range(len(bands)))
axes[0, 1].set_yticklabels(bands)
plt.colorbar(im2, ax=axes[0, 1])

# Difference matrix
im3 = axes[1, 0].imshow(diff_matrix, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
axes[1, 0].set_title('Difference (Train - Competition)')
axes[1, 0].set_xticks(range(len(bands)))
axes[0, 1].set_xticklabels(bands, rotation=45, ha='right')
axes[1, 0].set_yticks(range(len(bands)))
axes[1, 0].set_yticklabels(bands)
plt.colorbar(im3, ax=axes[1, 0])

# Add text annotations showing the values on the difference matrix
for i in range(len(bands)):
    for j in range(len(bands)):
        if not np.isnan(diff_matrix[i, j]):
            axes[1, 0].text(j, i, f'{diff_matrix[i, j]:.2f}', 
                           ha='center', va='center', 
                           color='white' if abs(diff_matrix[i, j]) > 0.25 else 'black',
                           fontsize=8)

# Scatter plot: train vs test correlation values
axes[1, 1].scatter(train_flat, test_flat, alpha=0.6, s=20)
axes[1, 1].plot([-1, 1], [-1, 1], 'r--', alpha=0.5, label='Perfect agreement (y=x)')
axes[1, 1].set_xlabel('Training Correlation')
axes[1, 1].set_ylabel('Competition Correlation')
axes[1, 1].set_title(f'Train vs Test Correlation Values\n(Matrix correlation = {matrix_correlation:.3f})')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print some key statistics
print("=== MATRIX STATISTICS ===")
print(f"Training matrix - Mean: {np.nanmean(train_corr_matrix):.4f}, Std: {np.nanstd(train_corr_matrix):.4f}")
print(f"Competition matrix - Mean: {np.nanmean(test_corr_matrix):.4f}, Std: {np.nanstd(test_corr_matrix):.4f}")
print(f"Difference matrix - Mean: {np.nanmean(diff_matrix):.4f}, Std: {np.nanstd(diff_matrix):.4f}")
print()

# Analyze diagonal vs off-diagonal elements
diag_train = np.diag(train_corr_matrix)
off_diag_train = train_corr_matrix[~np.eye(len(bands), dtype=bool)]
diag_test = np.diag(test_corr_matrix)
off_diag_test = test_corr_matrix[~np.eye(len(bands), dtype=bool)]

print("Diagonal elements (should be ~1.0):")
print(f"  Training - Mean: {np.nanmean(diag_train):.4f}")
print(f"  Competition - Mean: {np.nanmean(diag_test):.4f}")
print()
print("Off-diagonal elements:")
print(f"  Training - Mean: {np.nanmean(off_diag_train):.4f}, Std: {np.nanstd(off_diag_train):.4f}")
print(f"  Competition - Mean: {np.nanmean(off_diag_test):.4f}, Std: {np.nanstd(off_diag_test):.4f}")
print()

# SAR-specific analysis: check if SAR bands show different correlation patterns
sar_indices = [i for i, band in enumerate(bands) if band in ['VH', 'VV']]
optical_indices = [i for i, band in enumerate(bands) if band not in ['VH', 'VV']]

if len(sar_indices) >= 2:
    sar_corr_train = train_corr_matrix[np.ix_(sar_indices, sar_indices)]
    sar_corr_test = test_corr_matrix[np.ix_(sar_indices, sar_indices)]
    
    print("SAR-SPECIFIC ANALYSIS (VH, VV bands):")
    print(f"  Training SAR correlation - Mean: {np.nanmean(sar_corr_train):.4f}")
    print(f"  Competition SAR correlation - Mean: {np.nanmean(sar_corr_test):.4f}")
    if np.std(np.nan_to_num(sar_corr_train)) > 0 and np.std(np.nan_to_num(sar_corr_test)) > 0:
        sar_matrix_corr, _ = pearsonr(
            sar_corr_train[~np.isnan(sar_corr_train)], 
            sar_corr_test[~np.isnan(sar_corr_test)]
        )
        print(f"  SAR matrix correlation: {sar_matrix_corr:.4f}")
    print()

print("Step 7 complete. Results saved for subsequent steps.")
step7_results = {
    'train_corr_matrix': train_corr_matrix,
    'test_corr_matrix': test_corr_matrix,
    'diff_matrix': diff_matrix,
    'matrix_correlation': matrix_correlation,
    'frobenius_norm': frobenius_norm,
    'mean_absolute_error': mean_absolute_error,
    'valid_elements_count': np.sum(valid_elements)
}

In [ ]:
# Step 8: Synthesize and provide final verdict
print("=== STEP 8: SYNTHESIS AND FINAL VERDICT ===")
print()

print("Walking through each test's result and what it does/doesn't support.")
print("Giving one overall calibrated verdict.")
print()

# Collect all the results from previous steps
print("=== SUMMARY OF FINDINGS ===")
print()

# Step 0: Metadata
print("Step 0 - Metadata Scan:")
print("  ✓ File sizes and basic structure verified")
print(f"  ✓ Training: {len(train_df):,} rows, {len(train_df.columns)} columns")
print(f"  ✓ Competition: {len(test_df):,} rows, {len(test_df.columns)} columns")
print()

# Step 1: Load & characterize
print("Step 1 - Load & Characterize:")
print(f"  ✓ Training rows: {len(train_df_features):,}")
print(f"  ✓ Competition rows: {len(test_df_features):,}")
print(f"  ✓ Average months observed - Train: {np.mean([np.sum(row > 0) for row in step6_results['train_patterns']]):.1f}")
print(f"  ✓ Average months observed - Test: {np.mean([np.sum(row > 0) for row in step6_results['test_patterns']]):.1f}")
print()

# Step 2: Aggregate cyclic-shift scan
print("Step 2 - Aggregate Cyclic-Shift Scan:")
print(f"  ✓ Best shift τ*: {step2_results['best_tau']}")
print(f"  ✓ Best average Pearson correlation: {step2_results['best_avg_pearson']:.4f}")
print(f"  ✓ Best average Spearman correlation: {step2_results['spearman_scores'][step2_results['best_tau']]:.4f}")
print()

# Step 3: Calibrate against baselines
print("Step 3 - Baseline Calibration:")
print(f"  ✓ Same-distribution ceiling (training halves): {step3_results['same_dist_best_pearson']:.4f}")
print(f"  ✓ Unrelated floor mean: {step3_results['unrelated_pearson_mean']:.4f}")
print(f"  ✓ Empirical p-value (Pearson): {step3_results['p_value_pearson']:.4f}")
print(f"  ✓ Empirical p-value (Spearman): {step3_results['p_value_spearman']:.4f}")
print()

# Step 4: Per-band affine fit
print("Step 4 - Per-Band Affine Fit Quality:")
if 'affine_results' in step4_results:
    r_squared_vals = [r['r_squared'] for r in step4_results['affine_results'] if not np.isnan(r['r_squared']) and r['n_points'] >= 2]
    if r_squared_vals:
        print(f"  ✓ Mean R² across bands: {np.mean(r_squared_vals):.4f}")
        print(f"  ✓ Median R² across bands: {np.median(r_squared_vals):.4f}")
        print(f"  ✓ Bands with R² > 0.7: {sum(1 for r in r_squared_vals if r > 0.7)}/{len(r_squared_vals)}")
    # SAR-specific
    sar_bands = ['VH', 'VV']
    sar_results = [r for r in step4_results['affine_results'] if r['band'] in sar_bands and r['n_points'] >= 2]
    if sar_results:
        print(f"  ✓ SAR bands analysis:")
        for sar_result in sar_results:
            print(f"    {sar_result['band']}: slope={sar_result['slope_a']:.3f}, intercept={sar_result['intercept_b']:.3f}, R²={sar_result['r_squared']:.3f}")
print()

# Step 5: Row-level correspondence search
print("Step 5 - Row-Level Correspondence Search:")
if 'correlations_array' in step5_results and len(step5_results['correlations_array']) > 0:
    corr_array = step5_results['correlations_array']
    print(f"  ✓ Valid matches: {len(step5_results['valid_matches'])}/{len(test_df_features)}")
    print(f"  ✓ Mean best correlation: {np.mean(corr_array):.4f}")
    print(f"  ✓ Median best correlation: {np.median(corr_array):.4f}")
    print(f"  ✓ Top correlation: {np.max(corr_array):.4f}")
    if len(step5_results['null_correlations_array']) > 0:
        null_array = step5_results['null_correlations_array']
        print(f"  ✓ Null distribution mean: {np.mean(null_array):.4f}")
        exceed_count = np.sum(null_array >= np.max(corr_array)) if len(corr_array) > 0 else 0
        print(f"  ✓ Empirical p-value (top match): {exceed_count/len(null_array):.4f}")
print()

# Step 6: Missingness-pattern fingerprint
print("Step 6 - Missingness-Pattern Fingerprint:")
print(f"  ✓ Fixed shift τ*={step6_results['best_tau']} match rate: {step6_results['actual_match_rate']:.4f}")
print(f"  ✓ Per-row optimal shift match rate: {step6_results['per_row_match_rate']:.4f}")
print(f"  ✓ Empirical p-value (fixed shift): {step6_results['empirical_p_value']:.4f}")
print(f"  ✓ Empirical p-value (per-row shift): {step6_results['empirical_p_value_per_row']:.4f}")
print()

# Step 7: Inter-band correlation structure preservation
print("Step 7 - Inter-Band Correlation Structure:")
print(f"  ✓ Matrix correlation: {step7_results['matrix_correlation']:.4f}")
print(f"  ✓ Frobenius norm: {step7_results['frobenius_norm']:.4f}")
print(f"  ✓ Mean absolute error: {step7_results['mean_absolute_error']:.4f}")
print()

print("=== EVIDENCE SYNTHESIS ===")
print()

# Count evidence for H1 vs H0
evidence_for_H1 = 0
evidence_for_H0 = 0
inconclusive = 0

print("Evidence assessment:")
print()

# Step 2 evidence
if step2_results['best_avg_pearson'] > 0.7:
    evidence_for_H1 += 1
    print("✅ Step 2 (Aggregate cyclic-shift): STRONG evidence for H1")
    print(f"   High correlation ({step2_results['best_avg_pearson']:.3f}) at shift τ*={step2_results['best_tau']}")
elif step2_results['best_avg_pearson'] > 0.3:
    inconclusive += 1
    print("⚠️  Step 2 (Aggregate cyclic-shift): WEAK evidence for H1")
    print(f"   Moderate correlation ({step2_results['best_avg_pearson']:.3f}) at shift τ*={step2_results['best_tau']}")
else:
    evidence_for_H0 += 1
    print("❌ Step 2 (Aggregate cyclic-shift): EVIDENCE for H0")
    print(f"   Low correlation ({step2_results['best_avg_pearson']:.3f}) suggests different origins")
print()

# Step 3 evidence
if step3_results['p_value_pearson'] < 0.01:
    evidence_for_H1 += 1
    print("✅ Step 3 (Baseline calibration): STRONG evidence for H1")
    print(f"   Real result significantly exceeds unrelated baseline (p = {step3_results['p_value_pearson']:.4f})")
elif step3_results['p_value_pearson'] < 0.05:
    evidence_for_H1 += 1
    print("✅ Step 3 (Baseline calibration): MODERATE evidence for H1")
    print(f"   Real result exceeds unrelated baseline (p = {step3_results['p_value_pearson']:.4f})")
elif step3_results['p_value_pearson'] < 0.10:
    inconclusive += 1
    print("⚠️  Step 3 (Baseline calibration): WEAK evidence for H1")
    print(f"   Real result somewhat exceeds unrelated baseline (p = {step3_results['p_value_pearson']:.4f})")
else:
    evidence_for_H0 += 1
    print("❌ Step 3 (Baseline calibration): EVIDENCE for H0")
    print(f"   Real result not significantly different from unrelated baseline (p = {step3_results['p_value_pearson']:.4f})")
print()

# Step 4 evidence
if 'affine_results' in step4_results:
    r_squared_vals = [r['r_squared'] for r in step4_results['affine_results'] if not np.isnan(r['r_squared']) and r['n_points'] >= 2]
    if r_squared_vals:
        mean_r2 = np.mean(r_squared_vals)
        if mean_r2 > 0.7:
            evidence_for_H1 += 1
            print("✅ Step 4 (Per-band affine fit): STRONG evidence for H1")
            print(f"   High affine fit quality (mean R² = {mean_r2:.3f})")
        elif mean_r2 > 0.3:
            inconclusive += 1
            print("⚠️  Step 4 (Per-band affine fit): WEAK evidence for H1")
            print(f"   Moderate affine fit quality (mean R² = {mean_r2:.3f})")
        else:
            evidence_for_H0 += 1
            print("❌ Step 4 (Per-band affine fit): EVIDENCE for H0")
            print(f"   Poor affine fit quality (mean R² = {mean_r2:.3f})")
print()

# Step 5 evidence
if 'correlations_array' in step5_results and len(step5_results['correlations_array']) > 0:
    corr_array = step5_results['correlations_array']
    if len(step5_results['null_correlations_array']) > 0:
        null_array = step5_results['null_correlations_array']
        if len(corr_array) > 0:
            top_corr = np.max(corr_array)
            exceed_count = np.sum(null_array >= top_corr)
            p_val_top = exceed_count / len(null_array)
            
            if p_val_top < 0.01:
                evidence_for_H1 += 1
                print("✅ Step 5 (Row-level correspondence): STRONG evidence for H1")
                print(f"   Top match correlation ({top_corr:.3f}) highly unlikely by chance (p = {p_val_top:.4f})")
            elif p_val_top < 0.05:
                evidence_for_H1 += 1
                print("✅ Step 5 (Row-level correspondence): MODERATE evidence for H1")
                print(f"   Top match correlation ({top_corr:.3f}) unlikely by chance (p = {p_val_top:.4f})")
            elif p_val_top < 0.10:
                inconclusive += 1
                print("⚠️  Step 5 (Row-level correspondence): WEAK evidence for H1")
                print(f"   Top match correlation ({top_corr:.3f}) somewhat unlikely by chance (p = {p_val_top:.4f})")
            else:
                evidence_for_H0 += 1
                print("❌ Step 5 (Row-level correspondence): EVIDENCE for H0")
                print(f"   Top match correlation ({top_corr:.3d}) consistent with chance (p = {p_val_top:.4f})")
print()

# Step 6 evidence
if step6_results['empirical_p_value'] < 0.01:
    evidence_for_H1 += 1
    print("✅ Step 6 (Missingness-pattern): STRONG evidence for H1")
    print(f"   Missingness patterns match beyond chance (p = {step6_results['empirical_p_value']:.4f})")
elif step6_results['empirical_p_value'] < 0.05:
    evidence_for_H1 += 1
    print("✅ Step 6 (Missingness-pattern): MODERATE evidence for H1")
    print(f"   Missingness patterns match beyond chance (p = {step6_results['empirical_p_value']:.4f})")
elif step6_results['empirical_p_value'] < 0.10:
    inconclusive += 1
    print("⚠️  Step 6 (Missingness-pattern): WEAK evidence for H1")
    print(f"   Missingness patterns somewhat match beyond chance (p = {step6_results['empirical_p_value']:.4f})")
else:
    evidence_for_H0 += 1
    print("❌ Step 6 (Missingness-pattern): EVIDENCE for H0")
    print(f"   Missingness patterns consistent with chance (p = {step6_results['empirical_p_value']:.4f})")
print()

# Step 7 evidence
matrix_corr = step7_results['matrix_correlation']
if not np.isnan(matrix_corr):
    if matrix_corr > 0.8:
        evidence_for_H1 += 1
        print("✅ Step 7 (Inter-band correlation): STRONG evidence for H1")
        print(f"   Band correlation structures highly similar (r = {matrix_corr:.3f})")
    elif matrix_corr > 0.5:
        evidence_for_H1 += 1
        print("✅ Step 7 (Inter-band correlation): MODERATE evidence for H1")
        print(f"   Band correlation structures moderately similar (r = {matrix_corr:.3f})")
    elif matrix_corr > 0.3:
        inconclusive += 1
        print("⚠️  Step 7 (Inter-band correlation): WEAK evidence for H1")
        print(f"   Band correlation structures somewhat similar (r = {matrix_corr:.3f})")
    else:
        evidence_for_H0 += 1
        print("❌ Step 7 (Inter-band correlation): EVIDENCE for H0")
        print(f"   Band correlation structures dissimilar (r = {matrix_corr:.3f})")
print()

print()
print("=== EVIDENCE SUMMARY ===")
print(f"Evidence for H1 (disguised same-origin): {evidence_for_H1}")
print(f"Evidence for H0 (independent regions): {evidence_for_H0}")
print(f"Inconclusive/Weak evidence: {inconclusive}")
print()

# Overall verdict
if evidence_for_H1 > evidence_for_H0 and evidence_for_H1 >= 3:
    verdict = "STRONG evidence for H1 (disguised same-origin data)"
    confidence = "High"
elif evidence_for_H1 > evidence_for_H0:
    verdict = "MODERATE evidence for H1 (disguised same-origin data)"
    confidence = "Medium"
elif evidence_for_H0 > evidence_for_H1 and evidence_for_H0 >= 3:
    verdict = "STRONG evidence for H0 (independent regions)"
    confidence = "High"
elif evidence_for_H0 > evidence_for_H1:
    verdict = "MODERATE evidence for H0 (independent regions)"
    confidence = "Medium"
else:
    verdict = "INCONCLUSIVE - evidence is balanced or insufficient"
    confidence = "Low"

print("=== OVERALL VERDICT ===")
print(f"🎯 {verdict}")
print(f"Confidence level: {confidence}")
print()

# List concrete caveats
print("=== CONCRETE CAVEATS ===")
print()

if step5_results.get('use_approximate', False):
    print("⚠️  Step 5 used approximate nearest-neighbor search for efficiency")
    print("   • This may miss some true matches if signature method is not perfect")
    print("   • K-value used: {}".format(min(50, max(1, len(train_df_features)//20))))
else:
    print("✅ Step 5 used full pairwise search")
    print("   • Computationally exhaustive but may have been slow for large datasets")

print(f"⚠️  Missingness pattern analysis used {len(step6_results['baseline_match_rates'])} permutations")
print("   • More permutations would give more precise p-values")
print("   • Assumes that shuffling destroys all real correspondence while preserving marginals")

print(f"⚠️  Baseline calibrations used random splits/shuffles")
print("   • Same-distribution ceiling based on single random split of training data")
print("   • Unrelated floor based on shuffled competition data")

if 'affine_results' in step4_results:
    low_data_bands = [r['band'] for r in step4_results['affine_results'] if r['n_points'] < 2]
    if low_data_bands:
        print(f"⚠️  Step 4: Insufficient data for affine fit in bands: {low_data_bands}")

print()
print("=== RECOVERED TRANSFORM PARAMETERS (IF H1 SUPPORTED) ===")
print()

if evidence_for_H1 > evidence_for_H0:
    print("If H1 is supported, the estimated transform parameters are:")
    print(f"  • Month shift (τ*): {step2_results['best_tau']} months")
    print("    (Competition month = Training month + τ* mod 12)")
    print()
    print("  • Per-band affine parameters (competition = a × train + b):")
    print("    Band\t\tSlope (a)\tIntercept (b)")
    print("    --------\t--------\t---------")
    if 'affine_results' in step4_results:
        for result in step4_results['affine_results']:
            if result['n_points'] >= 2 and not np.isnan(result['slope_a']):
                print(f"    {result['band']:<8}\t{result['slope_a']:.4f}\t\t{result['intercept_b']:.4f}")
            else:
                print(f"    {result['band']:<8}\t{'N/A':<8}\t\t{'N/A':<8}")
    print()
    print("To apply the transform to align competition to training:")
    print("  1. Shift competition month indices by -τ* (comp_month - τ*) mod 12")
    print("  2. For each band, apply: train_estimate = (comp_value - b) / a")
    print()
else:
    print("H1 not sufficiently supported - transform parameters not reliable")
    print("Best available estimates from analysis:")
    print(f"  • Month shift (τ*): {step2_results['best_tau']} months")
    if 'affine_results' in step4_results:
        print("  • Per-band affine parameters:")
        for result in step4_results['affine_results']:
            if result['n_points'] >= 2 and not np.isnan(result['slope_a']):
                print(f"    {result['band']}: slope={result['slope_a']:.3f}, intercept={result['intercept_b']:.3f}")

print()
print("=== LIMITATIONS AND ASSUMPTIONS ===")
print()
print("Assumptions made in this analysis:")
print("  1. The transform, if present, is consistent across all rows (global shift)")
print("  2. Affine transformation is appropriate per band (value' = a_b × value + b_b)")
print("  3. Missing data is random with respect to the underlying signal")
print("  4. -9999 values represent missing data and are handled consistently")
print("  5. Month indexing is consistent (01-12 representing Jan-Dec)")
print()
print("Limitations:")
print("  1. Row-level search may have used approximations for efficiency")
print("  2. Baseline calibrations use finite samples (splits, shuffles)")
print("  3. Correlation-based methods may miss non-linear but monotonic relationships")
print("  4. Does not account for potential noise added after transformation")
print("  5. Assumes temporal stationarity of band relationships")
print()

print("Step 8 complete. Analysis finished.")
print()
print("NOTEBOOK COMPLETE")